*0.1 Python for GenAI*

# poetry

**The situation.** You join a team whose repositories, CI pipelines and Dockerfiles are all built around Poetry. Adding uv on the side gives two lock files that disagree within a week.

**The fix: same idea, their tool.** Poetry keeps the same two files — `pyproject.toml` for what you want, `poetry.lock` for what was resolved. The commands differ; the guarantee is the same. One tool per repository.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The same project, with Poetry.** Run through `uvx` so nothing is installed globally. One detail: Poetry refuses to work inside another tool's activated environment, so the notebook's `VIRTUAL_ENV` variable is removed for these commands.

In [2]:
import os
import subprocess
import tempfile
import tomllib
from pathlib import Path

env = dict(os.environ)
env.pop("VIRTUAL_ENV", None)
folder = tempfile.mkdtemp()
subprocess.run(
    ["uvx", "poetry", "new", "support-bot"], cwd=folder, capture_output=True, check=True, env=env
)
project = Path(folder) / "support-bot"
subprocess.run(
    ["uvx", "poetry", "add", "httpx", "--lock"],
    cwd=project,
    capture_output=True,
    check=True,
    env=env,
)

pyproject = tomllib.loads((project / "pyproject.toml").read_text())
lock = tomllib.loads((project / "poetry.lock").read_text())
print("you asked for:  ", pyproject["project"]["dependencies"])
print("poetry.lock pinned:", len(lock["package"]), "packages")
print("CI installs with: poetry install --sync")
assert len(lock["package"]) > 3

you asked for:   ['httpx (>=0.28.1,<0.29.0)']
poetry.lock pinned: 7 packages
CI installs with: poetry install --sync


**Reading the output.** Same result as uv: a declared dependency and a lock file pinning every version.

```
uv add httpx        ◀──▶  poetry add httpx
uv.lock             ◀──▶  poetry.lock
uv sync --frozen    ◀──▶  poetry install --sync
```

| Use it when | Don't when | Instead use |
|---|---|---|
| the repository already uses it | starting fresh — uv is faster | uv |

**Watch out**
- Never two managers in one repo.
- `poetry install --sync` in CI removes packages that are no longer in the lock.
- `poetry export -f requirements.txt` for systems that only understand pip.